# Music Recommender — LightGBM Ranking Model Training

**What this notebook does (top to bottom):**

| Step | What happens |
|---|---|
| 1 | Mount Google Drive and point at your project folder |
| 2 | Load `songs_full.csv` and `interactions.csv` |
| 3 | Precompute PCA song + artist embeddings (same logic as `src/embeddings/precompute.py`) |
| 4 | For every `(user, song)` pair in interactions, compute 6 features |
| 5 | Split by user into train (80%) / test (20%) |
| 6 | Train a LightGBM binary classifier to predict like (1) vs skip (0) |
| 7 | Evaluate with Precision@K, Recall@K, NDCG@K |
| 8 | Save the model to `models/ranking_model.pkl` and download it |

**Why run on Colab?**
The local machine cannot install `pandas` or `lightgbm` (MINGW Python, no MSVC compiler).
Colab provides a Linux environment with these packages pre-installed and ~12 GB RAM.

**Why LightGBM?**
LightGBM is a gradient-boosted tree model. Unlike neural networks it:
- Trains in minutes on a CPU
- Automatically discovers feature interactions (e.g. "high freshness AND high similarity")
- Returns probability scores (`predict_proba`) used directly as ranking scores
- Requires no GPU — the default Colab runtime is enough

---
## Step 1 — Install packages and mount Google Drive

In [ ]:
# lightgbm is the only package Colab does not pre-install.
# scikit-learn, numpy, and pandas come with Colab by default.
!pip install lightgbm --quiet
print("lightgbm installed.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH to match where you placed the project on your Drive ──
REPO_PATH = '/content/drive/MyDrive/applied-ai-music-recommendation-system'

import os
if not os.path.isdir(REPO_PATH):
    raise FileNotFoundError(
        f"Project not found at {REPO_PATH}\n"
        "Upload the project folder to Google Drive and update REPO_PATH above."
    )
print(f"Project found at: {REPO_PATH}")

---
## Step 2 — Imports and configuration

In [ ]:
import csv
import json
import pickle
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# ── paths ─────────────────────────────────────────────────────────────────────
REPO             = Path(REPO_PATH)
DATA_DIR         = REPO / 'data'
EMBEDDINGS_DIR   = REPO / 'embeddings'
MODELS_DIR       = REPO / 'models'
SONGS_CSV        = DATA_DIR / 'songs_full.csv'
INTERACTIONS_CSV = DATA_DIR / 'interactions.csv'

EMBEDDINGS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# ── constants — must match src/embeddings/precompute.py and src/ranking/features.py ──
AUDIO_FEATURES = [
    'energy', 'valence', 'danceability', 'acousticness',
    'instrumentalness', 'speechiness', 'liveness',
    'tempo_bpm', 'loudness', 'popularity_norm',
]

N_COMPONENTS = 32    # PCA output dimension
DECAY_LAMBDA = 0.01  # freshness exponential decay constant

# Column ORDER matters — ranker.py must pass features in this exact order.
FEATURE_COLUMNS = [
    'content_similarity',
    'artist_similarity',
    'freshness',
    'popularity',
    'collaborative_score',
    'diversity_penalty',
]


def safe_float(value) -> float:
    """Parse a value to float; return 0.0 on failure or None."""
    try:
        return float(value) if value is not None and value != '' else 0.0
    except (ValueError, TypeError):
        return 0.0


print("Imports OK.")
print(f"  Songs CSV:        {SONGS_CSV}")
print(f"  Interactions CSV: {INTERACTIONS_CSV}")

---
## Step 3 — Load songs and interactions

In [ ]:
# Load songs into a DataFrame and a dict keyed by song_id for O(1) lookup.
songs_df = pd.read_csv(SONGS_CSV, encoding='utf-8', encoding_errors='replace')
print(f"Songs loaded: {len(songs_df):,} rows, {songs_df.shape[1]} columns")

songs_by_id = {row['song_id']: row for row in songs_df.to_dict('records')}

# Schema: user_id, song_id, label (1=liked / 0=skipped), play_count, timestamp
interactions_df = pd.read_csv(INTERACTIONS_CSV, encoding='utf-8')
interactions_df['label'] = interactions_df['label'].astype(int)

print(f"\nInteractions loaded: {len(interactions_df):,} rows")
print(f"  Label distribution: {interactions_df['label'].value_counts().to_dict()}")
print(f"  Unique users: {interactions_df['user_id'].nunique():,}")
print(f"  Unique songs: {interactions_df['song_id'].nunique():,}")

---
## Step 3 — Precompute PCA embeddings

This replicates `src/embeddings/precompute.py`. We inline the logic here so the notebook
is self-contained and does not need to import from your local modules.

**What PCA does:**
Think of 10 audio features as 10 directions in space. PCA finds 32 new directions that
capture the most variation in how songs differ from each other. Songs that sound similar
end up close together in this 32-dimensional space, so cosine distance becomes a meaningful
"sounds alike" score.

**Why StandardScaler first:**
`tempo_bpm` ranges 0-250; `energy` ranges 0-1. Without scaling, PCA would devote most of
its axes to explaining tempo variance. Scaling puts every feature on equal footing (mean=0,
std=1) before PCA runs.

In [ ]:
print("Building raw feature matrix for PCA ...")
feature_rows = [
    [safe_float(row.get(feat)) for feat in AUDIO_FEATURES]
    for row in songs_df.to_dict('records')
]
X_raw = np.array(feature_rows, dtype=np.float32)
print(f"  Feature matrix shape: {X_raw.shape}")

print("Running StandardScaler + PCA ...")
X_scaled = StandardScaler().fit_transform(X_raw)
n_comp = min(N_COMPONENTS, X_raw.shape[1])
pca = PCA(n_components=n_comp, random_state=42)
song_embeddings = pca.fit_transform(X_scaled).astype(np.float32)
print(f"  Explained variance retained: {pca.explained_variance_ratio_.sum():.1%}")
print(f"  Song embeddings shape: {song_embeddings.shape}")

# song_id <-> row-index maps
song_id_to_index = {}
index_to_song_id = {}
for idx, song in enumerate(songs_df.to_dict('records')):
    sid = song['song_id']
    song_id_to_index[sid] = idx
    index_to_song_id[str(idx)] = sid

# Artist embedding = mean of all that artist's song embeddings.
# It summarises "what does this artist's music typically sound like?"
artist_song_indices = defaultdict(list)
for idx, song in enumerate(songs_df.to_dict('records')):
    aid = str(song.get('artist_id', '') or '').strip()
    if aid:
        artist_song_indices[aid].append(idx)

artist_ids = sorted(artist_song_indices.keys())
artist_id_to_index = {}
index_to_artist_id = {}
artist_rows = []

for art_idx, artist_id in enumerate(artist_ids):
    indices = artist_song_indices[artist_id]
    artist_vec = song_embeddings[indices].mean(axis=0)
    artist_rows.append(artist_vec)
    artist_id_to_index[artist_id] = art_idx
    index_to_artist_id[str(art_idx)] = artist_id

artist_embeddings = np.array(artist_rows, dtype=np.float32)
print(f"  Artist embeddings shape: {artist_embeddings.shape}")

np.save(str(EMBEDDINGS_DIR / 'song_embeddings.npy'),   song_embeddings)
np.save(str(EMBEDDINGS_DIR / 'artist_embeddings.npy'), artist_embeddings)

embedding_index = {
    'song_id_to_index':   song_id_to_index,
    'index_to_song_id':   index_to_song_id,
    'artist_id_to_index': artist_id_to_index,
    'index_to_artist_id': index_to_artist_id,
}
with open(EMBEDDINGS_DIR / 'embedding_index.json', 'w', encoding='utf-8') as f:
    json.dump(embedding_index, f)

print(f"\nEmbeddings saved to {EMBEDDINGS_DIR}/")

---
## Step 4 — Cosine similarity helpers

Cosine similarity measures the angle between two vectors: 1.0 = identical direction
(very similar), 0.0 = perpendicular (no relation).

We use vectorised numpy operations (`matrix @ query`) instead of Python loops, so all
N candidates are scored in a single compiled C call — roughly 100x faster than a loop.

In [ ]:
def cosine_single(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors. Returns 0.0 if either is zero-length."""
    norm_a = float(np.linalg.norm(a))
    norm_b = float(np.linalg.norm(b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


def cosine_batch(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    Cosine similarity: query (D,) vs every row of matrix (N x D) -> shape (N,).

    `matrix @ query` computes all N dot products at once as a matrix-vector multiply.
    Zero-norm rows produce 0.0 safely (not NaN).
    """
    query_norm = float(np.linalg.norm(query))
    if query_norm == 0.0:
        return np.zeros(len(matrix), dtype=np.float32)
    row_norms  = np.linalg.norm(matrix, axis=1)
    safe_norms = np.where(row_norms == 0.0, 1.0, row_norms)
    dots = matrix @ query
    return (dots / (query_norm * safe_norms)).astype(np.float32)


print("Cosine helpers defined.")

---
## Step 5 — Build training feature matrix

For every `(user, song)` row in `interactions.csv` we compute the same 6 features
that `src/ranking/features.py` computes at inference time.

**Avoiding data leakage in `collaborative_score`:**
For a liked song, we exclude that song from the liked set before computing its score
(leave-one-out). Otherwise the model sees that a song is perfectly similar to itself
and learns a spurious signal that does not exist at inference time.

**Small simplification (acceptable for v1):**
The user embedding is built from ALL their liked songs, not just those liked before each
interaction's timestamp. A fully temporal approach prevents any future leakage but adds
significant complexity. We note it here so you know where a future improvement could go.

In [ ]:
training_rows = []
total_users = interactions_df['user_id'].nunique()

for i, (user_id, group) in enumerate(interactions_df.groupby('user_id')):
    if i % max(1, total_users // 10) == 0:
        print(f"  Processing user {i+1}/{total_users} ...")

    liked_ids = group[group['label'] == 1]['song_id'].tolist()

    # User embedding: mean of liked song PCA vectors.
    # Sits at the centroid of all songs the user liked in PCA space.
    liked_indices = [song_id_to_index[s] for s in liked_ids if s in song_id_to_index]
    user_emb = (
        song_embeddings[liked_indices].mean(axis=0)
        if liked_indices
        else np.zeros(N_COMPONENTS, dtype=np.float32)
    )

    # Artist vector: mean of liked songs' artist PCA vectors.
    liked_artist_ids = [
        str(songs_by_id[s].get('artist_id', '') or '').strip()
        for s in liked_ids if s in songs_by_id
    ]
    art_indices = [artist_id_to_index[a] for a in liked_artist_ids if a in artist_id_to_index]
    artist_vec = (
        artist_embeddings[art_indices].mean(axis=0)
        if art_indices
        else np.zeros(N_COMPONENTS, dtype=np.float32)
    )

    for _, irow in group.iterrows():
        sid   = irow['song_id']
        label = int(irow['label'])

        if sid not in song_id_to_index or sid not in songs_by_id:
            continue

        song_idx  = song_id_to_index[sid]
        song_emb  = song_embeddings[song_idx]
        song_data = songs_by_id[sid]

        # Feature 1: content_similarity
        content_sim = cosine_single(user_emb, song_emb)

        # Feature 2: artist_similarity
        aid = str(song_data.get('artist_id', '') or '').strip()
        art_row_idx = artist_id_to_index.get(aid)
        artist_sim = (
            cosine_single(artist_vec, artist_embeddings[art_row_idx])
            if art_row_idx is not None else 0.0
        )

        # Feature 3: freshness  exp(-0.01 * age_in_days)
        date_str = str(song_data.get('release_date', '') or '').strip()
        try:
            age_days  = max(0, (datetime.now() - datetime.strptime(date_str, '%Y-%m-%d')).days)
            freshness = float(np.exp(-DECAY_LAMBDA * age_days))
        except ValueError:
            freshness = 0.0

        # Feature 4: popularity
        popularity = safe_float(song_data.get('popularity_norm', 0.0))

        # Feature 5: collaborative_score  (leave-one-out to avoid leakage)
        collab_indices = [
            song_id_to_index[s] for s in liked_ids
            if s in song_id_to_index and s != sid
        ]
        if collab_indices:
            liked_embs   = song_embeddings[collab_indices]
            sims         = cosine_batch(song_emb, liked_embs)
            collab_score = float(sims.mean())
        else:
            collab_score = 0.0

        # Feature 6: diversity_penalty
        # Set to 0 during training — it encodes position in a ranked list,
        # which does not exist during training.
        diversity_penalty = 0.0

        training_rows.append({
            'user_id':             user_id,
            'song_id':             sid,
            'label':               label,
            'content_similarity':  content_sim,
            'artist_similarity':   artist_sim,
            'freshness':           freshness,
            'popularity':          popularity,
            'collaborative_score': collab_score,
            'diversity_penalty':   diversity_penalty,
        })

train_df = pd.DataFrame(training_rows)
print(f"\nTraining dataset: {len(train_df):,} rows, {train_df['user_id'].nunique()} users")
print(f"Label distribution:\n{train_df['label'].value_counts()}")
print("\nFeature statistics:")
train_df[FEATURE_COLUMNS].describe().round(3)

---
## Step 6 — Train / test split

We split by **user**, not by row. The model is evaluated on users it has never seen
during training — simulating how the ranker will behave for a genuinely new user.

If we split randomly by row, the model would see 80% of user A's interactions during
training and predict the remaining 20% — much easier, and not representative of the real task.

In [ ]:
user_ids = train_df['user_id'].values
n_users  = train_df['user_id'].nunique()

if n_users < 2:
    # Only one user in the dataset — fall back to a row-level split.
    # This happens when the app has only been used once.
    print("WARNING: only 1 user found. Falling back to row-level 80/20 split.")
    print("Collect more user sessions to enable a proper user-level evaluation.")
    n      = len(train_df)
    cutoff = int(n * 0.8)
    train_idx = np.arange(cutoff)
    test_idx  = np.arange(cutoff, n)
else:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(train_df, groups=user_ids))

X = train_df[FEATURE_COLUMNS].values.astype(np.float32)
y = train_df['label'].values.astype(int)

X_train, y_train = X[train_idx], y[train_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

print(f"Train: {len(X_train):,} rows  ({y_train.mean():.1%} positive)")
print(f"Test:  {len(X_test):,} rows   ({y_test.mean():.1%} positive)")

---
## Step 7 — Train the LightGBM model

**Key hyperparameters:**
- `n_estimators=200` — build up to 200 trees (early stopping may stop sooner)
- `learning_rate=0.05` — each tree corrects the previous one by 5%; slower = more robust
- `num_leaves=31` — max branching per tree; higher = more expressive model
- `class_weight='balanced'` — likes are rare vs skips; this prevents the model from
  defaulting to always predicting "skip"
- `early_stopping(20)` — stop if validation loss does not improve in 20 rounds

After training, **feature importances** show how much each feature contributed to
splitting nodes across all trees. High importance = the model relies on that feature heavily.

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight='balanced',
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20, verbose=False),
        lgb.log_evaluation(period=50),
    ],
)

print(f"\nBest iteration: {model.best_iteration_}")
print("\nFeature importances (gain = how much each feature reduces prediction error):")
max_imp = max(model.feature_importances_) if max(model.feature_importances_) > 0 else 1
for feat, imp in sorted(zip(FEATURE_COLUMNS, model.feature_importances_), key=lambda x: -x[1]):
    bar = '#' * int(imp / max_imp * 30)
    print(f"  {feat:<24} {bar}")

---
## Step 8 — Evaluate

Recommendation is evaluated differently from classification. We ask: did the model rank
liked songs above skipped songs for each user?

**Precision@K** — of the K songs ranked at the top, what fraction did the user actually like?
**Recall@K** — of all songs the user liked, what fraction appear in the top K?
**NDCG@K** — like Precision@K but gives more credit for putting liked songs earlier
(position 1 is worth more than position K — it uses a logarithmic discount).

We compute these **per user** then average, so a user with 20 likes and a user with 2 likes
count equally.

In [ ]:
def precision_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    top_k = np.argsort(scores)[::-1][:k]
    return float(y_true[top_k].mean())


def recall_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    total_positive = float(y_true.sum())
    if total_positive == 0:
        return 0.0
    top_k = np.argsort(scores)[::-1][:k]
    return float(y_true[top_k].sum()) / total_positive


def ndcg_at_k(y_true: np.ndarray, scores: np.ndarray, k: int) -> float:
    """
    Normalised Discounted Cumulative Gain at K.
    Rewards putting positive items earlier in the list.
    Position 1 contributes gain/log2(2); position K contributes gain/log2(K+1).
    """
    def dcg(labels: np.ndarray) -> float:
        discounts = np.log2(np.arange(2, len(labels) + 2))
        return float((labels / discounts).sum())

    top_k       = np.argsort(scores)[::-1][:k]
    ideal_order = np.sort(y_true)[::-1][:k]
    ideal_dcg   = dcg(ideal_order.astype(float))
    if ideal_dcg == 0:
        return 0.0
    return dcg(y_true[top_k].astype(float)) / ideal_dcg


K = 10
test_user_ids = train_df.iloc[test_idx]['user_id'].values
test_scores   = model.predict_proba(X_test)[:, 1]

p_list, r_list, ndcg_list = [], [], []

for uid in np.unique(test_user_ids):
    mask     = test_user_ids == uid
    u_labels = y_test[mask]
    u_scores = test_scores[mask]

    if mask.sum() < K or u_labels.sum() == 0:
        continue

    p_list.append(precision_at_k(u_labels, u_scores, K))
    r_list.append(recall_at_k(u_labels, u_scores, K))
    ndcg_list.append(ndcg_at_k(u_labels, u_scores, K))

print(f"Evaluation on {len(p_list)} test users (K={K}):")
if p_list:
    print(f"  Precision@{K}:  {np.mean(p_list):.3f}")
    print(f"  Recall@{K}:     {np.mean(r_list):.3f}")
    print(f"  NDCG@{K}:       {np.mean(ndcg_list):.3f}")
else:
    print("  No test users had enough interactions for per-user ranking metrics.")
    print("  Collect more sessions (10+ interactions each) to enable this evaluation.")

if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, test_scores)
    print(f"  AUC (global):   {auc:.3f}")
    print("\nAUC > 0.5 means the model ranks liked songs above skipped songs better than chance.")
    print("AUC = 1.0 is perfect; AUC = 0.5 is random.")
else:
    print("  AUC: skipped — test set has only one label class.")

---
## Step 9 — Save the model and download it

We use `pickle` to serialise the trained LightGBM object. Pickle is Python's built-in
way to freeze any object to disk so it can be loaded later without retraining.

We also save `ranking_model_meta.json` with the feature column names and their order.
`src/ranking/ranker.py` must pass features in exactly this order at inference time.

**After downloading:** place both files in `models/` in your local project folder.

In [ ]:
model_path = MODELS_DIR / 'ranking_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

metadata = {'feature_columns': FEATURE_COLUMNS, 'n_components': N_COMPONENTS}
with open(MODELS_DIR / 'ranking_model_meta.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved to:    {model_path}")
print(f"Metadata saved to: {MODELS_DIR / 'ranking_model_meta.json'}")

In [ ]:
from google.colab import files
files.download(str(MODELS_DIR / 'ranking_model.pkl'))
files.download(str(MODELS_DIR / 'ranking_model_meta.json'))
print("Download started. Place both files in your local models/ folder.")

---
## What comes next

Once `models/ranking_model.pkl` is in your local project:

**Step 11 — Build `src/ranking/ranker.py`**
```python
import pickle
with open('models/ranking_model.pkl', 'rb') as f:
    model = pickle.load(f)

scores = model.predict_proba(feature_matrix)[:, 1]  # probability of like
ranked = sorted(zip(song_ids, scores), key=lambda x: -x[1])
top_50 = ranked[:50]
```

**Step 12a — Build `src/rag/build_knowledge_base.py`**
Fetch Wikipedia artist bios, auto-generate song descriptions, embed with MiniMax
`embo-01`, and build a FAISS index. Run once offline.

**Step 12b — Build `src/rag/rag_layer.py`**
At inference: embed the user's free-text intent, retrieve 5 relevant documents from
FAISS, and pass those documents + ML top-50 to Claude for final ranking and explanations.

**Re-training:** re-run this notebook whenever `interactions.csv` has grown significantly
(e.g. after 10+ new user sessions). More data -> better signal -> better model.